# MoviBot Scraper — Google Colab (T4)

⚠️ **Le GPU T4 ne sert à rien ici.** Chrome, le HTML et SQLite utilisent le CPU, la RAM et le réseau. Ce notebook n'utilise **que le CPU** ; inutile de choisir un GPU, mais ça fonctionne quand même si vous en avez un.

**Limites de Colab à connaître** : session ~12 h maximum, disque `/content` éphémère, IP de datacenter (certains sites répondent 403). Le notebook contourne ça avec :
- un **checkpoint automatique de la base sur Google Drive** toutes les 100 fiches ;
- la **reprise** : si la session meurt, relancez simplement les cellules, le travail déjà fait est relu depuis Drive.

Pour un travail de ~5 jours, un petit VPS allumé en permanence reste plus simple. Colab demande ~10 sessions de 12 h.

In [ ]:
# 1) Monter Google Drive et préparer les chemins
from google.colab import drive
drive.mount('/content/drive')

import os, shutil, pathlib
DRIVE = pathlib.Path('/content/drive/MyDrive/movibot')
DRIVE.mkdir(parents=True, exist_ok=True)
DB_LOCAL = '/content/movibot-scraper/data/index_web.sqlite3'
DB_CHECKPOINT = str(DRIVE / 'index_web.sqlite3')
REPO = '/content/movibot-scraper'

# Reprise : si la base locale a disparu mais que le checkpoint Drive existe, on le recopie.
if not os.path.exists(DB_LOCAL) and os.path.exists(DB_CHECKPOINT):
    os.makedirs(os.path.dirname(DB_LOCAL), exist_ok=True)
    shutil.copy2(DB_CHECKPOINT, DB_LOCAL)
    print('✅ Base restaurée depuis Drive :', os.path.getsize(DB_LOCAL)//1024, 'Ko')
else:
    print('Base locale :', os.path.getsize(DB_LOCAL)//1024 if os.path.exists(DB_LOCAL) else 'aucune (nouveau départ)')
print('Checkpoint Drive :', DB_CHECKPOINT)

In [ ]:
# 2) Récupérer le dépôt privé
# Option A (recommandée) : stockez un token GitHub dans les SECRETS Colab (🔑), nommé GITHUB_TOKEN.
# Option B : uploadez le dossier movibot-scraper dans Drive/movibot/ et décommentez la copie.
import os, shutil
if os.path.isdir(REPO):
    print('✅ Dépôt déjà présent')
else:
    try:
        from google.colab import userdata
        token = userdata.get('GITHUB_TOKEN')
        os.system(f'git clone -q https://{token}@github.com/nemesishighschool-cloud/movibot-scraper.git {REPO}')
        print('✅ Clone depuis GitHub')
    except Exception as e:
        src = DRIVE / 'movibot-scraper'
        if src.is_dir():
            shutil.copytree(src, REPO)
            print('✅ Copie depuis Drive')
        else:
            raise RuntimeError("Ajoutez le secret GITHUB_TOKEN dans Colab (🔑) OU uploadez le dossier dans Drive/movibot/movibot-scraper") from e
os.chdir(REPO)
print('Répertoire :', os.getcwd())

In [ ]:
# 3) Installer les dépendances (2 à 3 min, une seule fois par session)
!python3 -m pip install -q -r requirements.txt
!python3 -m playwright install chromium
!python3 -m playwright install-deps chromium > /dev/null 2>&1 || true
!python3 -c "from playwright.sync_api import sync_playwright; p=sync_playwright().start(); b=p.chromium.launch(headless=True,args=['--no-sandbox']); print('✅ Chromium opérationnel'); b.close(); p.stop()"

In [ ]:
# 4) Configuration : 1 worker par source, checkpoint Drive, pauses par source
import os, pathlib, textwrap
cfg = f'''
SCRAPER_DB={DB_LOCAL}
SCRAPER_WORKERS=4
SCRAPER_CHECKPOINT={DB_CHECKPOINT}
SCRAPER_CHECKPOINT_EVERY=100
SCRAPER_MAX_PAGES=0
SCRAPER_INCREMENTAL=1
'''
pathlib.Path('config.env').write_text(cfg, encoding='utf-8')
os.chmod('config.env', 0o600)
print(cfg)
!python3 -m scraper.cli sources | head -30
!python3 -m scraper.cli estimate

In [ ]:
# 5) TEST RAPIDE : 3 fiches par source, 4 navigateurs en parallèle (≈1 min)
import os
os.environ['SCRAPER_LIMIT']='3'; os.environ['SCRAPER_MAX_PAGES']='2'; os.environ['SCRAPER_PAUSE']='1'
!python3 -m scraper.cli run --limit 3 --pause 1 --no-players
os.environ.pop('SCRAPER_LIMIT', None); os.environ.pop('SCRAPER_MAX_PAGES', None); os.environ.pop('SCRAPER_PAUSE', None)
!python3 -m scraper.cli status

In [ ]:
# 6) LANCER LA COLLECTE en tâche de fond (le notebook reste utilisable)
# Le scraper s'arrête proprement à la fin de la session ; relancez la cellule 1 puis celle-ci pour reprendre.
import os, subprocess, time, pathlib
log = pathlib.Path('/content/scraper.log')
env = dict(os.environ)
f = open(log, 'w')
p = subprocess.Popen(['python3','-u','-m','scraper.cli','run'], cwd=REPO, env=env,
                     stdout=f, stderr=subprocess.STDOUT)
print('PID', p.pid, '| log', log)
time.sleep(20)
print(log.read_text()[-2500:])

In [ ]:
# 7) Suivre, vérifier et exporter (à exécuter quand vous voulez)
!tail -n 25 /content/scraper.log
!python3 -m scraper.cli status
!python3 -m scraper.cli verify
!python3 -m scraper.cli export
import shutil, os
for f in ['exports/Media_web.csv','exports/MoviBot_Base_Web.xlsx']:
    if os.path.exists(f):
        shutil.copy2(f, DRIVE / os.path.basename(f))
        print('☁️ copié sur Drive :', os.path.basename(f))

## Après une déconnexion

1. Relancez la **cellule 1** : elle remonte Drive et recopie `index_web.sqlite3` depuis le checkpoint.
2. Relancez les cellules **2 → 4** (installation) puis **6**.
3. Le scraper reprend exactement là où il s'était arrêté : chaque fiche est marquée `done` en base, les autres restent `pending`. Aucun doublon.

## Réglages utiles

| Variable | Effet |
|---|---|
| `SCRAPER_WORKERS=4` | 1 Chrome par source (jamais 2 sur le même domaine) |
| `SCRAPER_ENABLE=monstream` | réactiver une source quand son domaine revient |
| `SCRAPER_DISABLE=wawacity` | exclure une source |
| `SCRAPER_MAX_PAGES=200` | borner Wawacity pour un test |
| `--no-players` | passe catalogue seule, ~2× plus rapide |

⚠️ **Révoquez le token GitHub** utilisé pour le clone une fois le dépôt copié sur Drive.